# Manipulação de Dados Tabulares com Pandas — Chamados de Suporte de TI

Sistemas de Informação • Ciência de Dados • Pandas

Nome: Cléder Rafael Narciso de Araujo

Contents

Exercício 1 — Reconhecendo o dataset

Exercício 2 — Tipos, ausências e resumo

Exercício 3 — Seleção

Exercício 4 — Filtros de negócio

In [ ]:
import pandas as pd

## Exercício 1 — Reconhecendo o dataset

Carregue o CSV. Exiba as cinco primeiras e as três últimas linhas. Informe dimensões e nomes das colunas.

In [ ]:
df = pd.read_csv("chamados_suporte_ti.csv")
df.head()

In [ ]:
df.tail(3)

In [ ]:
df.shape # Retorna uma tupla com o número de linhas e colunas

In [ ]:
df.columns # Retorna uma lista com o nome de todas as colunas

**Comentário:** o DataFrame `df` possui 18 linhas (chamados) e 10 colunas (variáveis), combinando identificação, categorização e desfecho de cada chamado.

## Exercício 2 — Tipos, ausências e resumo

Use info() e describe(). Quais colunas têm ausências? Qual tempo merece investigação por ser muito maior que a maioria?

In [ ]:
df.info() # Retorna vazio e imprime na tela as informações gerais do dataframe

In [ ]:
df.describe() # Retorna um objeto DataFrame com as medidas estatísticas para cada coluna numérica

**Comentário:** `tempo_resolucao_horas` e `satisfacao` têm apenas 16 valores não nulos (2 chamados ainda em aberto/andamento). O valor máximo de 120 h em `tempo_resolucao_horas` deve ser investigado, não removido automaticamente.

## Exercício 3 — Seleção

Selecione id, setor, prioridade e status; com posições, mostre cinco linhas e quatro colunas; explique seleção versus filtro.

In [ ]:
recorte = df[["id_chamado", "setor", "prioridade", "status"]]
recorte.head()

In [ ]:
posicional = df.iloc[0:5, 0:4]
posicional

**Comentário:** seleção escolhe variáveis (colunas); filtro escolhe observações (linhas) que atendem a uma condição.

## Exercício 4 — Filtros de negócio

Encontre prioridade Alta; resolvidos acima de 24 h; Rede ou Acesso. Conte cada resultado.

In [ ]:
altos = df[df["prioridade"] == "Alta"]
altos

In [ ]:
lentos = df[(df["status"] == "Resolvido") & (df["tempo_resolucao_horas"] > 24)]
lentos

In [ ]:
rede_acesso = df[df["categoria"].isin(["Rede", "Acesso"])]
rede_acesso

In [ ]:
print(altos.shape[0], lentos.shape[0], rede_acesso.shape[0]) # contagens dos três filtros

**Comentário:** 4 chamados de prioridade Alta; 4 chamados resolvidos com tempo acima de 24 h; 7 chamados nas categorias Rede ou Acesso. O filtro de tempo > 24h ajuda a identificar possíveis descumprimentos de SLA, e o de Rede/Acesso mostra concentração de demanda nessas duas categorias.

## Exercício 5 — Transformação e resumo

Converta a data, crie tempo_dias e calcule tempo médio por categoria em ordem decrescente. Interprete a maior média.

In [ ]:
df["data_abertura"] = pd.to_datetime(df["data_abertura"])
df["tempo_dias"] = df["tempo_resolucao_horas"] / 24
df[["id_chamado", "data_abertura", "tempo_resolucao_horas", "tempo_dias"]].head()

In [ ]:
media = df.groupby("categoria")["tempo_resolucao_horas"].mean().sort_values(ascending=False)
media

**Comentário:** `data_abertura` agora é datetime (permite ordenar, filtrar por período e calcular diferenças de datas). `tempo_dias` converte horas em dias para leitura mais intuitiva. A categoria **Sistema lento** tem a maior média (47,75 h), mas esse valor é fortemente influenciado pelo chamado de 120 h — sem ele, a média da categoria cairia bastante. Isso reforça que médias devem sempre ser lidas junto com uma checagem de outliers.

## Desafio — Painel de atenção do suporte

SLA: Crítica = 4h; Alta = 8h; Média = 24h; Baixa = 48h. Identifique onde o suporte precisa de atenção: inspecione, selecione/filtre resolvidos, crie `limite_sla_horas` e `dentro_sla`, calcule quantidade/percentual dentro e fora do SLA e o tempo médio por categoria. Conclua com evidências e recomendação.

In [ ]:
colunas = ["id_chamado", "categoria", "prioridade", "status",
           "tempo_resolucao_horas", "satisfacao", "reaberto"]
analise = df[colunas].copy()
resolvidos = analise[analise["status"] == "Resolvido"].copy()
resolvidos.head()

In [ ]:
limites = {"Crítica": 4, "Alta": 8, "Média": 24, "Baixa": 48}
resolvidos["limite_sla_horas"] = resolvidos["prioridade"].map(limites)
resolvidos["dentro_sla"] = resolvidos["tempo_resolucao_horas"] <= resolvidos["limite_sla_horas"]
resolvidos.head()

In [ ]:
contagem = resolvidos["dentro_sla"].value_counts()
percentual = resolvidos["dentro_sla"].value_counts(normalize=True).mul(100).round(1)
print(contagem)
print()
print(percentual)

In [ ]:
media_por_categoria = resolvidos.groupby("categoria")["tempo_resolucao_horas"].mean().sort_values(ascending=False).round(2)
media_por_categoria

### Conclusão

Dos 16 chamados resolvidos com tempo registrado, **11 (68,8%) ficaram dentro do SLA** e **5 (31,2%) estouraram o prazo** — quase 1 em cada 3 chamados. A categoria **Sistema lento** tem o maior tempo médio de resolução (47,75 h), puxado principalmente pelo chamado de 120 h, que é um outlier e merece investigação isolada (não deve ser removido, mas também não deve ser tratado como "normal" na análise de SLA).

**Recomendação:** priorizar a investigação da causa raiz dos chamados de "Sistema lento", já que essa categoria concentra o maior atraso médio e o maior outlier — possivelmente aponta para um problema recorrente de infraestrutura ou capacidade que está afetando o cumprimento do SLA.

Os 2 chamados ainda "Em andamento"/"Aberto" (CH005 e CH011) foram excluídos do cálculo de tempo porque ainda não têm desfecho — incluí-los distorceria a análise, já que não sabemos quanto tempo ainda vão levar.